In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

BASE = '/Users/abilindemann/projects/sa-housing-food-insecurity/data/processed/'

# Load ACS block groups — we need the GEOID and will compute centroids from lat/lon
acs = pd.read_csv(BASE + 'acs_bexar_block_groups.csv', dtype={'GEOID': str})

# Load grocery stores
grocery = pd.read_csv(BASE + 'grocery_stores_bexar.csv')

print(f"Block groups: {len(acs)}")
print(f"Grocery stores: {len(grocery)}")
print(f"\nGrocery columns: {grocery.columns.tolist()}")
print(grocery[['Store_Name','Store_Type','Latitude','Longitude']].head())

Block groups: 1139
Grocery stores: 85

Grocery columns: ['Store_Name', 'Store_Type', 'Store_Street_Address', 'City', 'State', 'Zip_Code', 'County', 'Latitude', 'Longitude']
                    Store_Name   Store_Type   Latitude  Longitude
0                      HEB 262  Supermarket  29.480915 -98.595490
1                      HEB 385  Supermarket  29.470903 -98.495605
2                      HEB 556  Supermarket  29.464632 -98.526077
3       CULEBRA MEAT MARKET 01  Supermarket  29.447973 -98.554283
4  LACKLAND AFB COMMISSARY 622  Supermarket  29.381905 -98.612213


In [3]:
import requests

# Pull block group centroids for Bexar County from the Census geocoding API
# This returns the internal point (centroid) for each block group
url = (
    'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/'
    'tigerWMS_ACS2023/MapServer/10/query'
)
params = {
    'where': "STATE='48' AND COUNTY='029'",
    'outFields': 'GEOID,INTPTLAT,INTPTLON',
    'f': 'json',
    'resultRecordCount': 2000,
}

resp = requests.get(url, params=params, timeout=60)
features = resp.json()['features']
centroids = pd.DataFrame([f['attributes'] for f in features])
centroids.columns = ['GEOID', 'lat', 'lon']
centroids['lat'] = centroids['lat'].astype(float)
centroids['lon'] = centroids['lon'].astype(float)

print(f"Centroids pulled: {len(centroids)}")
print(centroids.head())

Centroids pulled: 1139
          GEOID        lat        lon
0  480291411022  29.358612 -98.438743
1  480291411012  29.353622 -98.440409
2  480291411013  29.357424 -98.443660
3  480291411023  29.363322 -98.450354
4  480291411011  29.358035 -98.455578


In [5]:
# Convert degrees to approximate miles (at San Antonio's latitude)
# 1 degree lat ≈ 69 miles, 1 degree lon ≈ 59 miles at ~29.5°N
def latlon_to_xy(lat, lon):
    x = lon * 59.0
    y = lat * 69.0
    return x, y

# Build KD-tree from grocery store coordinates
g_x, g_y = latlon_to_xy(grocery['Latitude'].values, grocery['Longitude'].values)
tree = cKDTree(np.column_stack([g_x, g_y]))

# Query nearest store for each block group centroid
c_x, c_y = latlon_to_xy(centroids['lat'].values, centroids['lon'].values)
distances, indices = tree.query(np.column_stack([c_x, c_y]), k=1)

centroids['nearest_grocery_miles'] = distances
centroids['nearest_grocery_name'] = grocery['Store_Name'].values[indices]

print("Distance to nearest grocery store (miles):")
print(centroids['nearest_grocery_miles'].describe().round(3))
print(f"\nBlock groups within 0.5 miles: {(centroids['nearest_grocery_miles'] <= 0.5).sum()}")
print(f"Block groups within 1.0 miles:  {(centroids['nearest_grocery_miles'] <= 1.0).sum()}")
print(f"Block groups beyond 1.0 mile:   {(centroids['nearest_grocery_miles'] > 1.0).sum()}")

Distance to nearest grocery store (miles):
count    1139.000
mean        1.321
std         1.453
min         0.024
25%         0.628
50%         0.958
75%         1.484
max        13.772
Name: nearest_grocery_miles, dtype: float64

Block groups within 0.5 miles: 191
Block groups within 1.0 miles:  595
Block groups beyond 1.0 mile:   544


In [7]:
# Load USDA food desert data
usda = pd.read_csv(BASE + 'usda_food_desert_bexar.csv', dtype={'CensusTract': str})

print(f"USDA tracts: {len(usda)}")
print(f"USDA columns: {usda.columns.tolist()}")

USDA tracts: 365
USDA columns: ['CensusTract', 'State', 'County', 'LILATracts_1And10', 'LILATracts_halfAnd10', 'LowIncomeTracts', 'LA1and10', 'LAhalfand10', 'PovertyRate', 'MedianFamilyIncome', 'lapop1share']


In [9]:
# The block group GEOID is 12 digits: state(2) + county(3) + tract(6) + block group(1)
# The tract GEOID is 11 digits: state(2) + county(3) + tract(6)
# So we match on the first 11 characters of the block group GEOID

centroids['CensusTract'] = centroids['GEOID'].str[:11]

# Merge centroids with USDA tract-level data
merged = centroids.merge(usda[[
    'CensusTract', 'LILATracts_1And10', 'LILATracts_halfAnd10',
    'LowIncomeTracts', 'PovertyRate', 'MedianFamilyIncome', 'lapop1share'
]], on='CensusTract', how='left')

# Merge in housing scores
housing = pd.read_csv(BASE + 'housing_scores.csv', dtype={'GEOID': str})
merged = merged.merge(housing[['GEOID', 'housing_score', 'housing_insecurity_high',
                                'B19013_001E', 'poverty_rate', 'pct_hispanic']], 
                      on='GEOID', how='left')

print(f"Merged rows: {len(merged)}")
print(f"Nulls after merge:")
print(merged[['LILATracts_1And10', 'housing_score']].isnull().sum())
print(merged.head(3))

Merged rows: 1139
Nulls after merge:
LILATracts_1And10    68
housing_score        31
dtype: int64
          GEOID        lat        lon  nearest_grocery_miles  \
0  480291411022  29.358612 -98.438743               0.369001   
1  480291411012  29.353622 -98.440409               0.410645   
2  480291411013  29.357424 -98.443660               0.477055   

          nearest_grocery_name  CensusTract  LILATracts_1And10  \
0  Progreso LMMM San Antonio 6  48029141102                0.0   
1                      HEB 444  48029141101                1.0   
2  Progreso LMMM San Antonio 6  48029141101                1.0   

   LILATracts_halfAnd10  LowIncomeTracts  PovertyRate  MedianFamilyIncome  \
0                   1.0              1.0    17.829457             39044.0   
1                   1.0              1.0    41.006289             32464.0   
2                   1.0              1.0    41.006289             32464.0   

   lapop1share  housing_score  housing_insecurity_high  B19013_001E  \


In [11]:
# Food access score: based on distance to nearest grocery store
# Normalize distance so 0 = best access, 1 = worst access
def normalize(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn)

merged['food_distance_norm'] = normalize(merged['nearest_grocery_miles'])

# Binary access flags
merged['within_half_mile'] = (merged['nearest_grocery_miles'] <= 0.5).astype(int)
merged['within_one_mile']  = (merged['nearest_grocery_miles'] <= 1.0).astype(int)

# Food insecurity flag: beyond 1 mile OR USDA-designated food desert
merged['food_insecurity_high'] = (
    (merged['nearest_grocery_miles'] > 1.0) |
    (merged['LILATracts_1And10'] == 1.0)
).astype(int)

print(f"Block groups with poor food access (>1 mile): {(merged['nearest_grocery_miles'] > 1.0).sum()}")
print(f"Block groups in USDA food desert:             {(merged['LILATracts_1And10'] == 1.0).sum()}")
print(f"Block groups flagged high food insecurity:    {merged['food_insecurity_high'].sum()}")
print(f"\nFood distance score distribution:")
print(merged['food_distance_norm'].describe().round(3))

Block groups with poor food access (>1 mile): 544
Block groups in USDA food desert:             222
Block groups flagged high food insecurity:    616

Food distance score distribution:
count    1139.000
mean        0.094
std         0.106
min         0.000
25%         0.044
50%         0.068
75%         0.106
max         1.000
Name: food_distance_norm, dtype: float64


In [13]:
out_cols = [
    'GEOID', 'lat', 'lon',
    'nearest_grocery_miles', 'nearest_grocery_name',
    'food_distance_norm',
    'within_half_mile', 'within_one_mile',
    'LILATracts_1And10', 'LILATracts_halfAnd10', 'LowIncomeTracts',
    'lapop1share', 'PovertyRate', 'MedianFamilyIncome',
    'food_insecurity_high',
    'housing_score', 'housing_insecurity_high',
    'B19013_001E', 'poverty_rate', 'pct_hispanic',
]

out = merged[[c for c in out_cols if c in merged.columns]].copy()
out.to_csv(BASE + 'food_scores.csv', index=False)
print(f"Saved food_scores.csv with {len(out)} rows and {len(out.columns)} columns")

Saved food_scores.csv with 1139 rows and 20 columns


## Notebook 02 — Food Access Score

**Output:** `data/processed/food_scores.csv`

**Method:** Distance from each block group centroid to nearest full-service grocery store,
computed using a KD-tree over 85 SNAP-authorized grocery stores in Bexar County.
Two thresholds applied: 0.5 mile (strict) and 1.0 mile (USDA standard).
USDA Food Desert Atlas (LILATracts_1And10) included as a cross-check.

**Key findings:**
- 191 block groups within 0.5 miles of a grocery store
- 595 block groups within 1.0 mile
- 544 block groups beyond 1.0 mile (no grocery store within walking distance)
- 222 block groups in USDA-designated food deserts
- 616 block groups flagged high food insecurity (beyond 1 mile OR USDA food desert) — 54% of Bexar County

**Note:** USDA data is tract-level; 68 block groups had no USDA match (low population tracts).